# Capstone Project Debugging

This notebook will be used for testing and debugging of the Python code that the notebook will execute. As the project code grew, my desing philosophy shifted towards moving more of the complex code out of the notebook istelf, such that the notebook becomes an orchestrator that produces results, makes plots, and has descriptive text, but that heavy-lifting and boilerplate code is handled elsewhere.

In [ ]:
import pandas as pd

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory

## Config

No snapshot flags set → `build()` leaves all version counters unchanged.
This run will read from GCS but write nothing back.

In [ ]:
config = VersionConfig.load().build()
run    = PipelineRun(config)
stages = PipelineFactory.validate_current(config)

print('\nScenario:', stages.scenario)
print('raw_version   :', config.raw_version)
print('model_version :', config.model_version)

## First-time Setup: Create the Locked Validation Holdout

**Run this section once**, before the main validation flow below.
Skip it if `splits/validation_ids.json` already exists in GCS — `DataSplitter` will
detect the file and take the load path automatically.

The cell loads the raw data snapshot and runs `DataSplitter`, which will print a
split summary and prompt for confirmation before writing anything to GCS.

In [ ]:
# Holdout creation — DataSplitter takes the create path on first call,
# load path on all subsequent calls (idempotent after the first write).
run_init    = PipelineRun(config)
stages_init = PipelineFactory.validate_current(config)

stages_init.loader.run(run_init)
stages_init.splitter.run(run_init)

print('\nHoldout ready — proceed with the main validation flow below.')

---
## Main Validation Flow

Runs the full `validate_current` scenario against the locked holdout.

## Stage 1 — DataLoader (GCS)

Reads the parquet snapshot at `config.raw_version`. No BigQuery call.

In [ ]:
stages.loader.run(run)
run.summary()

## Stage 2 — DataSplitter

Loads the locked validation IDs from GCS (`splits/validation_ids.json`).
Re-splits remaining data into train/test for feature engineering context.

In [ ]:
stages.splitter.run(run)
run.summary()

## Stage 3 — FeatureEngineer

Fits scaler on df_train (real only), transforms all three splits.
Also captures `run.X_val_unscaled` so Validator can apply each model's own historical scaler.

In [ ]:
stages.feature_engineer.run(run)
run.summary()

## Stage 4 — ModelLoader

Auto-discovers all `models/<model_version>_*/` directories in GCS and loads each.

In [ ]:
stages.model_loader.run(run)
print('\nLoaded models:', list(run.models.keys()))

## Stage 5 — Validator

Evaluates each loaded model on the locked validation set (`X_val_unscaled` + each model's own scaler).

In [ ]:
stages.validator.run(run)

## Results

This is the **first validation-set evaluation** for this project — there is no prior
validation baseline in `data_analysis.ipynb` (only test-set scores were recorded there).

Cross-check model rank order and approximate AUC range against the test scores
from `data_analysis.ipynb`. Validation AUC will typically be slightly lower than
test AUC since the holdout was never touched during training.

In [ ]:
metric_cols = [
    'roc_auc', 'accuracy',
    'f1_above', 'precision_above', 'recall_above',
    'f1_below', 'precision_below', 'recall_below',
]

df_results = (
    pd.DataFrame(run.results).T
    [metric_cols]
    .astype(float)
    .round(4)
)
df_results.index.name = 'model'

df_results.style.highlight_max(color='#d4edda').format('{:.4f}')

In [ ]:
# Top features for each model — sanity check against data_analysis.ipynb
for name, entry in run.results.items():
    top = entry.get('top_features', [])[:5]
    print(f'\n{name}:')
    for f in top:
        key = 'coefficient' if 'coefficient' in f else 'importance'
        print(f'  {f["feature"]:35s}  {f[key]:+.4f}')